# SR-KV on Kaggle

Thin entry point. **No algorithm logic lives in this notebook** - it clones the repo,
installs dependencies and calls `eval/run.py`. Everything else is in `src/` and `eval/`.

Work through the phases in order. Each cell is resumable: if the session dies, re-run
the same cell and it continues from where it stopped.

See `KAGGLE.md` for the full operational guide (session limits, sharding, troubleshooting).


## 0. Setup


In [ ]:
!nvidia-smi
import torch
print('cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')


In [ ]:
# Option A: clone from GitHub (edit the URL). Option B is in KAGGLE.md.
REPO = 'https://github.com/YOUR_USERNAME/sr-kv.git'
!git clone -q $REPO /kaggle/working/sr-kv || (cd /kaggle/working/sr-kv && git pull -q)
%cd /kaggle/working/sr-kv


In [ ]:
# transformers 5.x is required (see src/compat.py). torch is already in the image.
!pip install -q -U 'transformers>=5.0' accelerate bitsandbytes
import transformers; print('transformers', transformers.__version__)


In [ ]:
# carry forward results from a previous committed session, if attached as a data source
!mkdir -p results
!cp /kaggle/input/*/results/*.jsonl results/ 2>/dev/null || echo 'no previous results attached'
!ls -la results | head


## 1. Test suite (CPU, ~2 min, no downloads)

Run this first every session. It is the cheapest possible way to catch a broken change.


In [ ]:
!python -m pytest -q


## Phase 1 - harness sanity on a real model

**Pass condition: accuracy > 0.9.** A 1.5B instruct model finds a magic number in 512
tokens easily; anything less means the harness is broken, not the model.


In [ ]:
!python eval/run.py --method full --model qwen2.5-1.5b --task niah \
  --context_len 512 --depths 50 --n_samples 10 --max_new_tokens 16 \
  --output results/phase1_sanity.json


In [ ]:
# exercise the 4-bit fallback path at least once
!python eval/run.py --method full --model llama3.2-3b --precision 4bit --task niah \
  --context_len 512 --depths 50 --n_samples 3 \
  --output results/phase1_4bit.json


## Phase 2 - baselines

Qualitative check: StreamingLLM should fail at mid-sequence depths (it keeps only sinks
plus a recent window), SnapKV should hold up much better. Relative ordering, not exact
numbers from the papers.


In [ ]:
!python eval/run.py --method full,streaming_llm,snapkv --model qwen2.5-1.5b \
  --task niah --context_len 4096 --depths 0,25,50,75,100 --budget 0.3 \
  --n_samples 5 --output results/phase2_baselines.json


## Phase 3 - unified SR-KV class at 8k


In [ ]:
!python eval/run.py --method sr_kv,centroid_merge,snapkv_unified \
  --model qwen2.5-1.5b --task niah --context_len 8192 \
  --depths 0,50,100 --budget 0.3 --n_samples 3 \
  --output results/phase3_8k.json


## Phase 4 - RoPE position ablation, then freeze the winner

Do this **before** any large sweep. `freeze_rope_mode.py` writes the winner into
`configs/defaults.yaml`; every later phase inherits it from there.


In [ ]:
!make phase4 MODEL=qwen2.5-1.5b BUDGET=0.3 SAMPLES=5


In [ ]:
!python scripts/freeze_rope_mode.py --model qwen2.5-1.5b          # dry run
!python scripts/freeze_rope_mode.py --model qwen2.5-1.5b --apply
!cat configs/defaults.yaml


## Phase 5 - factorial matrix + LongBench

The long one. Prefer **Save Version -> Save & Run All (Commit)** so it survives a closed
browser. Add `SHARD=n NSHARDS=k` to split across accounts.


In [ ]:
!make phase5 MODEL=qwen2.5-1.5b BUDGET=0.3 SAMPLES=3


In [ ]:
!make phase5-longbench MODEL=qwen2.5-1.5b BUDGET=0.3


In [ ]:
!make check-complete MODEL=qwen2.5-1.5b BUDGET=0.3
!make check-ablation MODEL=qwen2.5-1.5b


## Phase 6 - hyperparameter sweep, then transfer to 3B

Sweep on 1.5B only. The 3B run uses the frozen 1.5B config unchanged - the question is
whether it transfers, so re-tuning there would answer a different question.


In [ ]:
!for a in 0.5 1.0 2.0; do for b in 0.0 0.3 0.6; do \
  python eval/run.py --method sr_kv --model qwen2.5-1.5b --task niah \
    --context_len 4096,8192 --depths 0,25,50,75,100 --budget 0.3 \
    --alpha $a --beta $b --n_samples 3 \
    --output results/phase6_sweep_a${a}_b${b}.json ; \
done; done


In [ ]:
!make phase6-3b BUDGET=0.3


## Phase 7 - all figures, one command


In [ ]:
!make report_artifacts
!ls -la figures


In [ ]:
from IPython.display import Image, display
import pathlib
for p in sorted(pathlib.Path('figures').glob('*.png')):
    print(p.name)
    display(Image(filename=str(p)))


## Save results so the next session can resume

Use **Save Version -> Save & Run All (Commit)**. Then in the next notebook, add this
notebook's output as a data source; the setup cell above copies the `.jsonl` files back
and `eval/run.py` picks up where it left off.
